# Semana 15 — Ranking Metrics: Recall@K, MRR e NDCG
**Fase 3: Advanced RAG, Search e Reranking** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/03-Phase-3-Advanced-RAG/notebooks/semana_15_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-15-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/03-Phase-3-Advanced-RAG/docs/Semana-15-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_15_gabarito.ipynb`.

In [ ]:
!pip -q install rank_bm25

In [ ]:
import numpy as np, pandas as pd, re, matplotlib.pyplot as plt
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ndcg_score
# --- Corpus de demonstração: base de conhecimento fictícia de uma empresa ---
import random
def corpus_demo(seed=7):
    """Devolve (docs, consultas).
    docs: lista de {id, texto, area, ano}.
    consultas: lista de {q, resposta, relevantes: {doc_id: grau}}; grau 2 = contém a resposta, 1 = mesmo assunto."""
    rnd = random.Random(seed)
    fatos = {
        "ferias": [("O colaborador tem direito a 30 dias de férias após 12 meses de trabalho.", "Quantos dias de descanso anual eu tenho?"),
                   ("As férias podem ser divididas em até três períodos, um deles com no mínimo 14 dias.", "Posso fracionar meu período de descanso?"),
                   ("O pedido de férias deve ser feito no portal RH com 45 dias de antecedência.", "Com que antecedência solicito minha folga anual?")],
        "reembolso": [("Despesas de viagem são reembolsadas em até 10 dias úteis após o envio dos recibos.", "Em quanto tempo recebo de volta o que gastei viajando?"),
                      ("O limite diário de alimentação em viagens nacionais é de R$ 120.", "Qual o teto de gasto com refeição em deslocamento a trabalho?"),
                      ("Corridas de aplicativo só são reembolsadas entre 22h e 6h ou em deslocamentos a clientes.", "A empresa paga meu Uber?")],
        "seguranca": [("Senhas corporativas expiram a cada 90 dias e exigem autenticação em dois fatores.", "De quanto em quanto tempo preciso trocar minha credencial?"),
                      ("Dados de clientes classificados como confidenciais não podem ser enviados a ferramentas externas de IA.", "Posso colar informações de cliente em um chatbot público?"),
                      ("Incidentes de segurança devem ser reportados ao SOC em até 2 horas.", "Qual o prazo para avisar sobre um vazamento?")],
        "ti": [("Notebooks são substituídos a cada 36 meses mediante chamado no service desk.", "Quando posso trocar meu computador?"),
               ("O acesso à VPN exige o cliente GlobalConnect e certificado pessoal.", "Como me conecto à rede da empresa de casa?"),
               ("Licenças de software adicionais precisam de aprovação do gestor e do time de compras.", "Como consigo um programa novo?")],
        "beneficios": [("O plano de saúde cobre dependentes legais sem custo adicional até 21 anos.", "Meu filho pode entrar no convênio médico?"),
                       ("O vale-refeição é creditado no último dia útil de cada mês.", "Em que dia cai o cartão de alimentação?"),
                       ("A empresa subsidia 50% de cursos de idiomas, limitado a R$ 400 por mês.", "Existe ajuda para estudar inglês?")],
    }
    codigos = [("O código de erro VPN-4031 indica certificado pessoal expirado; renove pelo portal de identidades.", "ti", "O que significa o erro VPN-4031?"),
               ("O formulário RH-207 é usado para solicitar licença não remunerada.", "ferias", "Para que serve o formulário RH-207?"),
               ("O centro de custo CC-8812 deve ser usado para despesas de treinamento externo.", "reembolso", "Qual centro de custo uso: CC-8812 é para quê?"),
               ("A política SEG-014 define a classificação de dados em público, interno e confidencial.", "seguranca", "O que diz a política SEG-014?")]
    projetos = ["Atlas", "Boreal", "Cometa", "Delta", "Éden"]
    armadilhas = ["O time do projeto {p} volta de férias em janeiro e retoma as entregas do trimestre.",
                  "No projeto {p}, o reembolso ao cliente foi aprovado após a revisão do contrato.",
                  "A equipe do projeto {p} trocou a senha do ambiente de testes durante a migração.",
                  "O projeto {p} comprou notebooks novos para o laboratório de dados.",
                  "O plano de saúde financeira do projeto {p} prevê corte de 10% nos custos de nuvem."]
    ruido = ["Este procedimento foi revisado pelo comitê interno.", "Dúvidas devem ser encaminhadas ao gestor imediato.",
             "A política vale para todas as unidades do grupo.", "Exceções exigem aprovação formal da diretoria."]
    docs, consultas = [], []
    def novo(texto, area):
        docs.append({"id": len(docs), "texto": texto, "area": area, "ano": rnd.choice([2023, 2024, 2025])}); return docs[-1]["id"]
    por_area = {}
    for area, itens in fatos.items():
        for fato, pergunta in itens:
            i = novo(f"{fato} {rnd.choice(ruido)}", area); por_area.setdefault(area, []).append(i)
            consultas.append({"q": pergunta, "resposta": fato, "relevantes": {i: 2}, "area": area})
    for texto, area, pergunta in codigos:
        i = novo(texto, area); por_area[area].append(i)
        consultas.append({"q": pergunta, "resposta": texto, "relevantes": {i: 2}, "area": area})
    for c in consultas:
        for j in por_area[c["area"]]:
            c["relevantes"].setdefault(j, 1)
    for p in projetos:
        for a in armadilhas:
            novo(a.format(p=p), "projetos")
    return docs, consultas

def manual_demo(docs):
    """Junta os documentos de política em um manual longo em Markdown (para exercícios de chunking)."""
    titulos = {"ferias": "Férias e licenças", "reembolso": "Viagens e reembolsos", "seguranca": "Segurança da informação",
               "ti": "Equipamentos e acesso", "beneficios": "Benefícios"}
    partes = ["# Manual do Colaborador\n\nEste manual reúne as políticas internas vigentes."]
    for area, titulo in titulos.items():
        frases = [d["texto"] for d in docs if d["area"] == area]
        partes.append(f"## {titulo}\n\n" + "\n\n".join(" ".join(frases[i:i + 2]) for i in range(0, len(frases), 2)))
    return "\n\n".join(partes)

docs, consultas = corpus_demo()
textos = [d["texto"] for d in docs]; print(len(docs), "docs |", len(consultas), "consultas")

## Passo 1 — Métricas em um exemplo à mão
`ranking` é a lista de ids devolvida pelo sistema. `rel` é o gabarito `{id: grau}`. Considere relevante qualquer grau ≥ 1.

In [ ]:
ranking = [7, 3, 9, 1, 5]
rel = {3: 2, 1: 1, 8: 1}          # o doc 8 é relevante, mas não foi recuperado

def precision_at_k(ranking, rel, k):
    # [SEU CÓDIGO AQUI] relevantes no top-k dividido por k

def recall_at_k(ranking, rel, k):
    # [SEU CÓDIGO AQUI] relevantes no top-k dividido pelo total de relevantes

def reciprocal_rank(ranking, rel, grau_min=1):
    # [SEU CÓDIGO AQUI] 1/posição do primeiro documento com grau >= grau_min (0 se não houver)

assert precision_at_k(ranking, rel, 5) == 0.4
assert abs(recall_at_k(ranking, rel, 5) - 2 / 3) < 1e-9
assert reciprocal_rank(ranking, rel) == 0.5
print("P@5 =", precision_at_k(ranking, rel, 5), "| R@5 =", round(recall_at_k(ranking, rel, 5), 3), "| RR =", reciprocal_rank(ranking, rel))

## Passo 2 — DCG e NDCG
`DCG@k = Σ rel_i / log2(i + 1)` com `i` começando em 1. O `IDCG` é o DCG da melhor ordenação possível dos graus do gabarito.

In [ ]:
def dcg_at_k(graus, k):
    # [SEU CÓDIGO AQUI] soma dos graus descontados por log2(posição + 1)

def ndcg_at_k(ranking, rel, k):
    # [SEU CÓDIGO AQUI] DCG do ranking dividido pelo DCG ideal

print("NDCG@5 =", round(ndcg_at_k(ranking, rel, 5), 4))
# conferência com o scikit-learn (mesma definição: ganho linear)
ids = sorted(set(ranking) | set(rel)); y_true = [[rel.get(i, 0) for i in ids]]
y_score = [[len(ranking) - ranking.index(i) if i in ranking else 0 for i in ids]]
assert abs(ndcg_at_k(ranking, rel, 5) - ndcg_score(y_true, y_score, k=5)) < 1e-9; print("confere com sklearn.metrics.ndcg_score")

## Passo 3 — A ordem importa
Mesmos documentos, ordens diferentes. Precision e Recall não mudam; MRR e NDCG sim.

In [ ]:
for nome, r in [("resposta exata no topo", [3, 1, 7, 9, 5]), ("relacionado no topo", [1, 3, 7, 9, 5]), ("relevantes no fim", [7, 9, 5, 1, 3])]:
    print(f"{nome:24s} | P@5={precision_at_k(r, rel, 5):.2f} R@5={recall_at_k(r, rel, 5):.2f} RR={reciprocal_rank(r, rel):.2f} NDCG@5={ndcg_at_k(r, rel, 5):.3f}")

## Passo 4 — Três sistemas de busca
BM25 por palavras, TF-IDF por n-gramas de caracteres (tolera variações de escrita) e a fusion dos dois por RRF.

In [ ]:
tok = lambda t: re.findall(r"\w+", t.lower())
bm25 = BM25Okapi([tok(t) for t in textos])
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(textos); M = tfidf.transform(textos)

def s_bm25(q, k=10): return [int(i) for i in np.argsort(-bm25.get_scores(tok(q)))[:k]]
def s_ngram(q, k=10): return [int(i) for i in np.argsort(-(M @ tfidf.transform([q]).T).toarray().ravel())[:k]]
def s_hibrido(q, k=10):
    pts = {}
    for lista in (s_bm25(q, 20), s_ngram(q, 20)):
        for pos, d in enumerate(lista, start=1): pts[d] = pts.get(d, 0) + 1 / (60 + pos)
    return sorted(pts, key=pts.get, reverse=True)[:k]

sistemas = {"BM25": s_bm25, "N-gramas": s_ngram, "Híbrido": s_hibrido}

## Passo 5 — Avaliação por query e agregada
Monte uma função que devolve, para um sistema, um DataFrame com uma linha por query e as colunas `R@3`, `R@10`, `MRR` (grau mínimo 2) e `NDCG@10`.

In [ ]:
def avaliar(fn):
    linhas = []
    for c in consultas:
        r = fn(c["q"], 10)
        # [SEU CÓDIGO AQUI] adicione em `linhas` um dict com as quatro métricas
    return pd.DataFrame(linhas)

por_consulta = {nome: avaliar(fn) for nome, fn in sistemas.items()}
resumo = pd.DataFrame({nome: df.mean() for nome, df in por_consulta.items()}).T.round(3); resumo

## Passo 6 — Recall@K em função de K
Essa curva responde: quantos candidatos preciso passar ao próximo stage?

In [ ]:
for nome, fn in sistemas.items():
    # [SEU CÓDIGO AQUI] recall médio para k de 1 a 20
    plt.plot(range(1, 21), curva, label=nome)
plt.xlabel("K"); plt.ylabel("Recall@K"); plt.legend(); plt.grid(alpha=.3); plt.show()

## Passo 7 — A diferença é real? Bootstrap pareado
Reamostre as queries com reposição 5.000 vezes; em cada reamostra, calcule a diferença média de NDCG@10 entre dois sistemas. Reporte o intervalo de 95%.

In [ ]:
def bootstrap_diff(a, b, n=5000, seed=0):
    rng = np.random.default_rng(seed); a, b = np.asarray(a), np.asarray(b)
    # [SEU CÓDIGO AQUI] devolva (diferença média, (limite inferior, limite superior))

for x, y in [("Híbrido", "BM25"), ("Híbrido", "N-gramas"), ("BM25", "N-gramas")]:
    m, (lo, hi) = bootstrap_diff(por_consulta[x]["NDCG@10"], por_consulta[y]["NDCG@10"])
    print(f"{x} − {y}: {m:+.3f}  IC95% [{lo:+.3f}, {hi:+.3f}]  {'significativo' if lo > 0 or hi < 0 else 'inconclusivo'}")

Com apenas 19 queries os intervalos são largos. Quantas queries anotadas você precisaria para detectar uma diferença de 0,03?

## Passo 8 — Onde cada sistema falha
Métricas agregadas escondem padrões. Liste as queries em que o melhor sistema tem MRR menor que 1.

In [ ]:
melhor = resumo["NDCG@10"].idxmax(); df = por_consulta[melhor]
# [SEU CÓDIGO AQUI] índices das queries com MRR < 1
for i in falhas:
    c = consultas[i]; r = sistemas[melhor](c["q"], 3)
    print(f"\n[{melhor}] MRR={df.loc[i, 'MRR']:.2f} | {c['q']}\n   esperado : {c['resposta'][:80]}\n   1º obtido: {textos[r[0]][:80]}")

**Reflexão:** as falhas têm algo em comum? (Dica: compare o vocabulário da pergunta com o do documento esperado.) É o argumento para o dense retrieval da Semana 14.